In [1]:
import sys
sys.path.append('/work/ConditionalDETR')

In [2]:
import torch
from models import build_model

device = 'cuda:1'

checkpoint = torch.load(f'../output/5kind50000img500epoch/partialV2_resize1/checkpoint0499.pth', map_location='cpu')
args = checkpoint['args']
args.num_classes = 20
model, criterion, postprocessors = build_model(args)
model.to(device)

model_without_ddp = model
model_without_ddp.load_state_dict(checkpoint['model'])
model_without_ddp
model.eval()
criterion.eval()
# checkpoint['model']
args

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth

Namespace(lr=0.0001, lr_backbone=1e-05, batch_size=8, weight_decay=0.0001, epochs=500, lr_drop=40, clip_max_norm=0.1, frozen_weights=None, backbone='resnet50', dilation=False, position_embedding='sine', enc_layers=6, dec_layers=6, dim_feedforward=2048, hidden_dim=256, dropout=0.1, nheads=8, num_queries=300, num_classes=20, pre_norm=False, masks=False, aux_loss=True, set_cost_class=2, set_cost_bbox=5, set_cost_giou=2, mask_loss_coef=1, dice_loss_coef=1, cls_loss_coef=2, bbox_loss_coef=5, giou_loss_coef=2, focal_alpha=0.25, dataset_file='coco', coco_path='/work', coco_panoptic_path=None, remove_difficult=False, output_dir='output/5kind50000img500epoch/partialV2_resize1', device='cuda:1', seed=42, resume='', start_epoch=0, eval=False, num_workers=2, world_size=1, dist_url='env://', distributed=False)

In [3]:
import torchvision.transforms as T

# 画像の前処理
preprocess = T.Compose([
    T.Resize(256),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 結果を格納するリスト
results = []

def box_cxcywh_to_xyxy(x):
    x_c, y_c, w, h = x.unbind(1)
    b = [(x_c - 0.5 * w), (y_c - 0.5 * h),
         (x_c + 0.5 * w), (y_c + 0.5 * h)]
    return torch.stack(b, dim=1)

def rescale_bboxes(out_bbox, size):
    img_w, img_h = size
    b = box_cxcywh_to_xyxy(out_bbox)
    b = b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32, device=out_bbox.device)
    return b


In [12]:
import os
from PIL import Image
from glob import glob
from utils import probas_to_scores_and_classes, nms, sort_by_bboxes

def probas_to_scores_and_classes(probas):
    scores = []
    classes = []
    for p in probas:
        cl = p.argmax().cpu().detach().numpy()
        score = p[cl].cpu().detach().numpy()
        scores.append(score)
        classes.append(cl)
    return scores, classes

pred_score = 0.7

flw_glob_path = '../../create_syntheic_flower_img/data/synthetic_flw/5kind_each10000img/flw/*/*'
image_paths = sorted(glob(flw_glob_path))
# image_paths = image_paths[:100]
print('images length {}'.format(len(image_paths)))

results = []

for i, img_path in enumerate(image_paths):
    if (i % 100) == 0:
        print('process {}'.format(i))
    # 画像を読み込み、前処理を行う
    img = Image.open(img_path).convert("RGB")  # RGBモードに変換
    img_tensor = preprocess(img).unsqueeze(0)  # バッチサイズの次元を追加

    # GPUが使える場合はGPUにデータとモデルを移す
    if torch.cuda.is_available():
        img_tensor = img_tensor.to(device)
        model.to(device)

    # モデルを使って予測を行う
    with torch.no_grad():  # 勾配計算を行わない
        outputs = model(img_tensor)

    # 予測結果からlogitsとボックスを取得
    probas = outputs['pred_logits'].softmax(-1)[0, :, :-1]
    bboxes = outputs['pred_boxes'][0]
    scaled_bboxes = rescale_bboxes(bboxes, img_tensor.shape[2:])
    
    keep = probas.max(-1).values > pred_score  # 信頼度が閾値を超えるボックスを保持

    keep_probas = [score for score, k in zip(probas, keep) if k]
    keep_bboxes = [bbox.cpu().detach().numpy().tolist() for bbox, k in zip(scaled_bboxes, keep) if k]

    scores, classes = probas_to_scores_and_classes(keep_probas)
    
    # nmsでbounding boxの重なりを削除
    filtered_bboxes, filtered_scores, filtered_classes = nms(keep_bboxes, scores, classes, 0.0001)

    # bounding boxの順番を時計回りに並び替え
    sorted_bboxes, sorted_scores, sorted_classes = sort_by_bboxes(filtered_bboxes, filtered_scores, filtered_classes)
    
    # 結果を保存
    results.append((img_path, [int(cl) for cl in sorted_classes], sorted_scores , sorted_bboxes))

images length 10
process 0


In [ ]:
# resultsをjsonファイルに保存
data = {}

for img_path, classes, scores, bboxes in results:
    # フォルダ名とファイル名を取得
    folder_name = os.path.basename(os.path.dirname(img_path))
    file_name = os.path.basename(img_path)
    
    # フォルダごとのデータ構造を作成
    if folder_name not in data:
        data[folder_name] = {}
        
    annotations = []
    
    for cls, score, bbox in zip(classes, scores, bboxes):
        x_min, y_min, x_max, y_max = bbox
        width = x_max - x_min
        height = y_max - y_min
        
        annotations.append({
            "category_id": cls,
            "bbox": [x_min, y_min, width, height],
            "score": float(score),
            "area": width * height,
            "iscrowd": 0
        })

    data[folder_name][file_name] = {
        "width": 256,
        "height": 256,
        "file_path": img_path.replace('../../', ''),
        "annotations": annotations
    }

# JSONファイルに保存
with open('./output/detection_results.json', 'w') as json_file:
    json.dump(data, json_file)


In [5]:
import matplotlib.pyplot as plt

CLASSES = [
    '0',
    '1',
    '2',
]

COLORS = [
    ("Red", "#FF0000", (255, 0, 0)),
    ("Green", "#00FF00", (0, 255, 0)),
    ("Blue", "#0000FF", (0, 0, 255)),
]

def plot_results(pil_img, classes, scores, boxes):
    plt.figure(figsize=(16,10))
    plt.imshow(pil_img)
    ax = plt.gca()
    for cl, score, (xmin, ymin, xmax, ymax) in zip(classes, scores, boxes):
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                   fill=False, color=COLORS[cl][1], linewidth=3))
        text = f'{CLASSES[cl]}: {score:0.2f}'
        ax.text(xmin, ymin, text, fontsize=15,
                bbox=dict(facecolor='yellow', alpha=0.5))
    plt.axis('off')
    plt.show()

In [6]:
# 花弁配置を0,1,2に変換
# 0:奥（白）, 1:中間（灰）, 2:手前（黒）
arranges = {'a1':[2,0,2,0],
    'a2':[1,2,1,0],
    'a3':[1,1,2,0],
    'b1':[1,0,2,0,2],
    'c1':[1,0,1,2,0,2],
    'c2':[0,2,0,2,0,2],
    'c3':[2,1,0,2,0,1],
    'd1':[2,1,0,2,1,0,1],
    'd2':[0,2,0,1,2,0,2],
    'e1':[2,1,0,1,2,0,2,0],
    'e2':[0,2,0,2,0,2,0,2],
    'e3':[0,2,0,2,1,0,2,1],
    'f1':[0,2,0,2,0,2,0,1,2],
    'g1':[2,0,2,0,2,0,2,0,2,0]
}

def reverse_order(arr):
    return arr[::-1]

def calculate_circular_fit(cycle1, cycle2, isLog=False):
    len_cycle1 = len(cycle1)
    len_cycle2 = len(cycle2)

    # Create a distance matrix
    dp = [[0] * (len_cycle2 + 1) for _ in range(len_cycle1 + 1)]

    # Initialize the distance matrix
    for i in range(len_cycle1 + 1):
        dp[i][0] = i  # Deletion
    for j in range(len_cycle2 + 1):
        dp[0][j] = j  # Insertion

    # Calculate the edit distance considering circular permutations
    for i in range(1, len_cycle1 + 1):
        for j in range(1, len_cycle2 + 1):
            if cycle1[i - 1] == cycle2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]  # No operation
            else:
                dp[i][j] = min(dp[i - 1][j] + 1,    # Deletion
                               dp[i][j - 1] + 1,    # Insertion
                               dp[i - 1][j - 1] + 1)  # Substitution

    # Check for circular shifts
    for shift in range(1, len_cycle2):
        shifted_cycle2 = cycle2[shift:] + cycle2[:shift]
        for i in range(1, len_cycle1 + 1):
            for j in range(1, len_cycle2 + 1):
                if cycle1[i - 1] == shifted_cycle2[j - 1]:
                    dp[i][j] = min(dp[i][j], dp[i - 1][j - 1])  # No operation
                else:
                    dp[i][j] = min(dp[i][j], 
                                   dp[i - 1][j] + 1,    # Deletion
                                   dp[i][j - 1] + 1,    # Insertion
                                   dp[i - 1][j - 1] + 1)  # Substitution

    edit_distance = dp[len_cycle1][len_cycle2]

    if isLog:
        print(f"配列1: {cycle1}, 配列2: {cycle2}, 編集距離: {edit_distance}")

    return edit_distance

def calculate_circular_fit_with_arrange(cycle1, cycle2, isLog=False):
    fit = calculate_circular_fit(cycle1, cycle2, isLog)
    reverse_fit = calculate_circular_fit(cycle1, reverse_order(cycle2), isLog)
    if (fit < reverse_fit):
        fit = reverse_fit
        cycle2 = reverse_order(cycle2)
    return fit, cycle2

def calculate_circular_fit_with_arranges(cycle1, isLog=False):
    max_fit = 0
    fit_arrange = []
    fit_arrange_key = ''
    for key, arrange_list in arranges.items():
        fit, arrange = calculate_circular_fit_with_arrange(cycle1, arrange_list, isLog)
        if (max_fit < fit):
            max_fit = fit
            fit_arrange = arrange
            fit_arrange_key = key
    
    return max_fit, fit_arrange, fit_arrange_key


In [ ]:
import json

# ./ground_truth.jsonを読み込む
with open('./output/ground_truth.json', 'r') as f:
    ground_truth = json.load(f)

def get_ground_truth(filename):
    # ファイル名を取得
    file_name = os.path.basename(filename)
    folder_name = os.path.basename(os.path.dirname(filename))
    # ground_truth.jsonからfile_nameのアノテーションを取得
    annotations = ground_truth[folder_name][file_name]['annotations']

    # アノテーションのclasses, bboxesを取得
    annotation_classes = [ann['category_id'] for ann in annotations]
    annotation_bboxes = [ann['bbox'] for ann in annotations]
    return annotation_classes, annotation_bboxes

In [ ]:
# フォルダごとに結果を分割 (1フォルダあたり10000サンプル)
SAMPLES_PER_FOLDER = 10000
each_folder_results = [
    results[i:i + SAMPLES_PER_FOLDER] 
    for i in range(0, len(results), SAMPLES_PER_FOLDER)
]

# フォルダ名を取得
folder_names = []
for filename, _, _, _ in results:
    folder_name = filename.split('/')[-2]  # パスからフォルダ名を抽出
    if folder_name not in folder_names:
        folder_names.append(folder_name)

print("フォルダ名一覧:")
for name in folder_names:
    print(name)

# 全フォルダの結果を格納する辞書
folder_stats = {}

# フォルダごとに結果を処理
for folder_idx, (folder_name, folder_results) in enumerate(zip(folder_names, each_folder_results)):
    folder_stats[folder_name] = {
        'total_stats': {
            'total_samples': 0,
            'correct_predictions': 0,
            'total_fit_score': 0,
            'total_fit_score_ground_truth': 0
        },
        'pattern_stats': {},
        'confusion_matrix': None,
        'others_patterns': [],
        'others_patterns_ground_truth': []
    }

    # カウンターの初期化
    counters = init_counters()
    heatmap_keys = sorted(list(arranges.keys()) + ['others'])
    key_to_idx = {key: i for i, key in enumerate(heatmap_keys)}
    confusion_matrix = np.zeros((len(heatmap_keys), len(heatmap_keys)))

    all_classes = []
    all_sorted_classes = []
    all_scores = []

    # 結果の処理
    for filename, classes, scores, bboxes in folder_results:
        folder_stats[folder_name]['total_stats']['total_samples'] += 1
        
        # 予測結果の処理
        max_fit_known, fit_arrange_known, fit_arrange_key_known = calculate_circular_fit_with_arranges(classes)
        folder_stats[folder_name]['total_stats']['total_fit_score'] += max_fit_known
        all_classes.append(classes)
        update_counters(counters, max_fit_known, fit_arrange_key_known, classes)

        # 正解データの処理
        annotation_classes, annotation_bboxes = get_ground_truth(filename)
        sorted_bboxes, _, sorted_classes = sort_by_bboxes(annotation_bboxes, [0]*len(annotation_bboxes), annotation_classes)
        all_sorted_classes.append(sorted_classes)
        all_scores.append(scores)

        max_fit_known_ground_truth, fit_arrange_known_ground_truth, fit_arrange_key_known_ground_truth = calculate_circular_fit_with_arranges(sorted_classes)
        folder_stats[folder_name]['total_stats']['total_fit_score_ground_truth'] += max_fit_known_ground_truth
        update_counters(counters, max_fit_known_ground_truth, fit_arrange_key_known_ground_truth, sorted_classes, True)

        # 混同行列の更新
        pred_key = fit_arrange_key_known if max_fit_known > 0 else 'others'
        true_key = fit_arrange_key_known_ground_truth if max_fit_known_ground_truth > 0 else 'others'
        confusion_matrix[key_to_idx[pred_key]][key_to_idx[true_key]] += 1

        # 正解判定
        if pred_key == true_key:
            folder_stats[folder_name]['total_stats']['correct_predictions'] += 1

        # マッチングカウンターの更新
        update_match_counters(counters, max_fit_known, max_fit_known_ground_truth, 
                            fit_arrange_key_known, fit_arrange_key_known_ground_truth)

    # フォルダごとの統計情報を保存
    folder_stats[folder_name]['pattern_stats'] = {
        key: {
            'matches': counters['key_match_counts'][key],
            'mismatches': counters['key_mismatch_counts'][key]
        } for key in arranges.keys()
    }
    folder_stats[folder_name]['confusion_matrix'] = confusion_matrix
    folder_stats[folder_name]['others_patterns'] = counters['others_patterns'][:10]
    folder_stats[folder_name]['others_patterns_ground_truth'] = counters['others_patterns_ground_truth'][:10]

# 結果の比較表示
print("\n=== フォルダ間の比較 ===")

# 1. 基本精度の比較
print("\n[1] 基本精度の比較")
print("フォルダ\t総サンプル数\t正解数\t正解率")
for folder_name, stats in folder_stats.items():
    total = stats['total_stats']['total_samples']
    correct = stats['total_stats']['correct_predictions']
    accuracy = (correct / total) * 100
    print(f"{folder_name}\t{total}\t{correct}\t{accuracy:.2f}%")

# 2. 適合度スコアの比較
print("\n[2] 適合度スコアの比較")
print("フォルダ\t予測平均\t正解平均")
for folder_name, stats in folder_stats.items():
    total = stats['total_stats']['total_samples']
    pred_avg = stats['total_stats']['total_fit_score'] / total
    true_avg = stats['total_stats']['total_fit_score_ground_truth'] / total
    print(f"{folder_name}\t{pred_avg:.2f}\t{true_avg:.2f}")

# 3. パターンごとの精度比較
print("\n[3] パターンごとの精度比較")
print("フォルダ", end="\t")
for key in arranges.keys():
    print(f"{key[:4]}", end="\t")
print()

for folder_name, stats in folder_stats.items():
    print(f"{folder_name}", end="\t")
    for key in arranges.keys():
        matches = stats['pattern_stats'][key]['matches']
        total = matches + stats['pattern_stats'][key]['mismatches']
        accuracy = (matches / total) * 100 if total > 0 else 0
        print(f"{accuracy:.1f}%", end="\t")
    print()

# 4. フォルダごとの詳細な結果表示と視覚化
for folder_name, stats in folder_stats.items():
    print(f"\n=== {folder_name} の詳細結果 ===")
    total = stats['total_stats']['total_samples']
    
    # 混同行列の表示
    print("\n混同行列:")
    print("予測＼正解", end="\t")
    for key in heatmap_keys:
        print(f"{key[:4]}", end="\t")
    print()
    
    confusion_matrix = stats['confusion_matrix']
    for i, pred_key in enumerate(heatmap_keys):
        print(f"{pred_key[:4]}", end="\t")
        for j in range(len(heatmap_keys)):
            print(f"{int(confusion_matrix[i][j])}", end="\t")
        print()

    # その他パターンの表示
    print("\nその他パターン (上位5件):")
    print("予測:", stats['others_patterns'][:5])
    print("正解:", stats['others_patterns_ground_truth'][:5])
    
    # グラフの描画
    plot_results_graphs(counters, confusion_matrix)